# Automatron E-commerce

Return disputes, chargeback packets, and seller appeal reviews.

## Contents

1. Constants and thresholds
2. Sample data
3. Return dispute tools
4. Chargeback tools
5. Seller appeal tools
6. Prompt addenda and workflows
7. Sector pack

Build the core module and put the generated package on the path. This cell is
for interactive use only and is dropped from the built module.

In [ ]:
import pathlib
import subprocess
import sys

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
subprocess.run([sys.executable, "scripts/build_notebooks.py"], check=True, cwd=ROOT)
sys.path.insert(0, str(ROOT / "automatron_build"))

In [ ]:
from automatron_core import *  # noqa: F401,F403

## 1. Constants and thresholds

Values read from config/sectors.yaml and the rule files, plus the masking helpers personal data passes through.

In [ ]:
import csv
import datetime as dt
import functools
import json
import pathlib
import re
from typing import Any

import yaml

SECTOR_ID = "ecommerce"
THRESHOLDS = sector_settings(SECTOR_ID)["thresholds"]

HIGH_VALUE_USD = float(THRESHOLDS["high_value_usd"])
RISK_HIGH = float(THRESHOLDS["risk_high"])
RISK_MEDIUM = float(THRESHOLDS["risk_medium"])

SAMPLE_DIR = get_settings().data_path / "samples" / SECTOR_ID
RULES_DIR = ROOT / "config" / "rules"

# Perceptual hash distance at or below which two images are treated as the same
# picture. Measured on the bundled set: the same image re-saved, recompressed or
# rescaled lands at 0 to 6, while different images land at 20 or more, so 10 sits
# in the empty gap between them. The distance is always reported alongside the
# verdict, because this is a signal a person checks rather than a finding.
IMAGE_MATCH_DISTANCE = 10

# Deterministic markers for the transcript check. Matching on phrasing is crude
# and is only ever a prompt to read the conversation, never a conclusion.
PRESSURE_PHRASES = (
    "right now", "immediately", "asap", "urgent", "escalate", "supervisor",
    "manager", "lawyer", "legal action", "chargeback", "social media", "review bomb",
    "last chance", "or else", "report you",
)
REFUND_DEMAND_PHRASES = (
    "full refund", "refund now", "money back", "want my money", "refund immediately",
)

SAMPLE_GENERATOR_VERSION = 2


@functools.lru_cache(maxsize=2)
def load_return_policy() -> dict[str, Any]:
    with (RULES_DIR / "return_policy.yaml").open(encoding="utf-8") as handle:
        return yaml.safe_load(handle)


@functools.lru_cache(maxsize=2)
def load_risk_weights() -> dict[str, Any]:
    with (RULES_DIR / "return_risk_weights.yaml").open(encoding="utf-8") as handle:
        return yaml.safe_load(handle)


@functools.lru_cache(maxsize=2)
def load_chargeback_rules() -> dict[str, Any]:
    with (RULES_DIR / "chargeback_evidence.yaml").open(encoding="utf-8") as handle:
        return yaml.safe_load(handle)


# --- personal data ---------------------------------------------------------------
# The core redaction pass masks logs and anything rendered. These mask at the tool
# boundary instead, so a full address or card number never enters a step result, an
# evidence entry, or a brief in the first place. Masking once it is already in the
# brief would be one missed code path away from leaking.

def mask_email(value: str) -> str:
    """first@domain -> f***@domain, keeping enough to recognise an address."""
    value = (value or "").strip()
    if "@" not in value:
        return value
    local, _, domain = value.partition("@")
    head = local[0] if local else ""
    return f"{head}***@{domain}"


def mask_phone(value: str) -> str:
    digits = re.sub(r"\D", "", value or "")
    return f"***{digits[-2:]}" if len(digits) >= 2 else "***"


def mask_card(value: str) -> str:
    """Last four only. The rest never leaves the sample file."""
    digits = re.sub(r"\D", "", value or "")
    return f"**** **** **** {digits[-4:]}" if len(digits) >= 4 else "****"


def mask_address(value: str) -> str:
    """Keep the locality, drop the street line: enough to compare, not to deliver."""
    parts = [p.strip() for p in (value or "").split(",") if p.strip()]
    return ", ".join(["[street withheld]", *parts[1:]]) if len(parts) > 1 else "[address withheld]"


def masked_customer(row: dict[str, str]) -> dict[str, Any]:
    """The customer fields a brief may carry, already masked."""
    return {
        "customer_id": row.get("customer_id", ""),
        "email": mask_email(row.get("email", "")),
        "phone": mask_phone(row.get("phone", "")),
        "country": row.get("country", ""),
        "account_opened": row.get("account_opened", ""),
    }

## 2. Sample data

ensure_samples(): invented customers, orders, claim photos,
dispute notices, sellers and invoices, all from a fixed seed.

In [ ]:
# Every person, order, seller and photograph here is invented and generated from a
# fixed seed. The images are coloured rectangles, not photographs of anything. Real
# claim data would carry personal information, which is exactly why none of this is
# real and why the tools mask what they return regardless.

SYNTHETIC_NOTE = ("Synthetic records generated for this project from a fixed seed. "
                  "Not real customers, orders, sellers or photographs.")

CATEGORIES = ("apparel", "electronics", "books", "grocery", "jewellery",
              "large_appliance", "personal_care", "custom_made", "perishable", "gift_card")
CARRIERS = ("Northbound Post", "Vantage Courier", "Harbour Freight Co")
COUNTRIES = ("US", "GB", "DE", "CA", "AU")
ITEM_NAMES = {
    "apparel": ("wool overshirt", "running jacket", "linen trousers", "knit scarf"),
    "electronics": ("wireless headphones", "portable monitor", "mesh router", "action camera"),
    "books": ("hardback novel", "field guide", "cookbook", "atlas"),
    "grocery": ("olive oil tin", "coffee beans", "spice set", "tea sampler"),
    "jewellery": ("silver pendant", "steel watch", "gold hoops", "signet ring"),
    "large_appliance": ("dishwasher", "tumble dryer", "range cooker", "chest freezer"),
    "personal_care": ("electric shaver", "hair dryer", "skincare set", "toothbrush head pack"),
    "custom_made": ("engraved tankard", "bespoke curtains", "custom desk mat", "tailored blazer"),
    "perishable": ("cheese selection", "fruit box", "fresh pasta", "bakery hamper"),
    "gift_card": ("gift card 50", "gift card 100", "gift card 25", "gift card 10"),
}

TODAY = dt.date(2026, 4, 12)


def _photo(seed: int, label: str, size=(320, 240)):
    """A deterministic stand-in for a claim photograph: coloured shapes, never a photo."""
    from PIL import Image, ImageDraw

    mixed = seed * 2654435761 % 2**32
    background = ((mixed >> 16) % 200 + 40, (mixed >> 8) % 200 + 40, mixed % 200 + 40)
    image = Image.new("RGB", size, background)
    draw = ImageDraw.Draw(image)
    for i in range(5):
        x0 = (mixed >> (i * 3)) % (size[0] // 2)
        y0 = (mixed >> (i * 5)) % (size[1] // 2)
        width = 40 + ((mixed >> (i * 7)) % 90)
        height = 30 + ((mixed >> (i * 11)) % 70)
        fill = ((mixed >> (i * 4)) % 256, (mixed >> (i * 6)) % 256, (mixed >> (i * 9)) % 256)
        draw.rectangle([x0, y0, x0 + width, y0 + height], fill=fill)
    draw.text((8, size[1] - 16), label, fill=(255, 255, 255))
    return image


def _save_photo(image, path: pathlib.Path, captured: dt.datetime | None,
                software: str = "SampleCam 1.0") -> None:
    """Write the photo, optionally with capture metadata in the PNG's exif chunk."""
    from PIL import Image

    if captured is None:
        image.save(path, format="PNG", optimize=True)
        return
    exif = Image.Exif()
    exif[0x0131] = software                                      # Software
    exif[0x0132] = captured.strftime("%Y:%m:%d %H:%M:%S")        # DateTime
    image.save(path, format="PNG", optimize=True, exif=exif)


def image_phash(path: pathlib.Path) -> str:
    import imagehash
    from PIL import Image

    with Image.open(path) as handle:
        return str(imagehash.phash(handle.convert("RGB")))


def hamming(left: str, right: str) -> int:
    """Distance between two hex perceptual hashes, in bits."""
    try:
        return bin(int(left, 16) ^ int(right, 16)).count("1")
    except ValueError:
        return 64


def _rng(seed: int):
    import random

    return random.Random(seed)


def _write_customers() -> list[dict[str, Any]]:
    rng = _rng(4711)
    rows = []
    for index in range(1, 61):
        opened = TODAY - dt.timedelta(days=rng.randint(40, 2200))
        orders = rng.randint(1, 30)
        returns = min(orders, max(0, int(rng.gauss(orders * 0.12, 1.5))))
        disputes = rng.choice([0, 0, 0, 0, 1, 1, 2])
        rows.append({
            "customer_id": f"CUST-{index:04d}",
            "email": f"sample.person{index}@example.invalid",
            "phone": f"+1555{1000000 + index * 137:07d}",
            "country": rng.choice(COUNTRIES),
            "account_opened": opened.isoformat(),
            "orders_count": orders,
            "returns_count": returns,
            "prior_disputes": disputes,
            "prior_disputes_upheld": min(disputes, rng.choice([0, 0, 1])),
        })

    # The three claim scenarios need specific histories, so pin those customers.
    by_id = {row["customer_id"]: row for row in rows}
    by_id["CUST-0007"].update({  # clean, long-standing
        "account_opened": (TODAY - dt.timedelta(days=800)).isoformat(),
        "orders_count": 14, "returns_count": 1, "prior_disputes": 0, "prior_disputes_upheld": 0})
    by_id["CUST-0031"].update({  # new, returns often, disputes upheld against them before
        "account_opened": (TODAY - dt.timedelta(days=20)).isoformat(),
        "orders_count": 6, "returns_count": 3, "prior_disputes": 2, "prior_disputes_upheld": 2})
    by_id["CUST-0019"].update({  # unremarkable history; the story is what differs
        "account_opened": (TODAY - dt.timedelta(days=400)).isoformat(),
        "orders_count": 9, "returns_count": 2, "prior_disputes": 0, "prior_disputes_upheld": 0})

    fields = list(rows[0])
    with (SAMPLE_DIR / "customers.csv").open("w", encoding="utf-8", newline="") as handle:
        handle.write(f"# {SYNTHETIC_NOTE}\n")
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    return rows


def _write_orders(customers: list[dict[str, Any]]) -> list[dict[str, Any]]:
    rng = _rng(90125)
    rows = []
    for index in range(200):
        order_id = f"ORD-{100000 + index:06d}"
        customer = rng.choice(customers)
        category = rng.choice(CATEGORIES)
        ordered = TODAY - dt.timedelta(days=rng.randint(3, 120))
        transit = rng.randint(2, 9)
        status = rng.choices(["delivered", "in_transit", "lost"], weights=[88, 9, 3])[0]
        delivered = ordered + dt.timedelta(days=transit) if status == "delivered" else None
        value = round(rng.choice([1, 1, 1, 2]) * rng.uniform(12, 420), 2)
        rows.append({
            "order_id": order_id,
            "customer_id": customer["customer_id"],
            "order_date": ordered.isoformat(),
            "expected_delivery": (ordered + dt.timedelta(days=transit)).isoformat(),
            "delivery_date": delivered.isoformat() if delivered else "",
            "delivery_status": status,
            "category": category,
            "item_name": rng.choice(ITEM_NAMES[category]),
            "quantity": rng.choice([1, 1, 1, 2, 3]),
            "value_usd": f"{value:.2f}",
            "currency": "USD",
            "carrier": rng.choice(CARRIERS),
            "tracking_number": f"TRK{rng.randint(10**9, 10**10 - 1)}",
            "shipping_address": (f"{rng.randint(1, 400)} Sample Street, Testville, "
                                 f"{customer['country']}"),
            "billing_matches_shipping": rng.choice(["yes", "yes", "yes", "no"]),
            "card_last4": f"{rng.randint(0, 9999):04d}",
            "avs_result": rng.choice(["full_match", "full_match", "partial_match", "no_match"]),
            "cvv_result": rng.choice(["match", "match", "match", "not_provided"]),
            "three_ds": rng.choice(["authenticated", "attempted", "not_used"]),
            "ip_match_prior_orders": rng.choice(["yes", "yes", "no"]),
            "device_match_prior_orders": rng.choice(["yes", "yes", "no"]),
            "delivery_confirmation": "signature" if status == "delivered" and rng.random() < 0.5
                                    else ("photo" if status == "delivered" else ""),
            "terms_accepted": "yes",
            "refund_issued_usd": "0.00",
        })

    pinned = {
        # Low value, clean, comfortably inside the window.
        "ORD-100042": {"customer_id": "CUST-0007", "category": "apparel",
                       "item_name": "wool overshirt", "value_usd": "45.00", "quantity": 1,
                       "order_date": (TODAY - dt.timedelta(days=9)).isoformat(),
                       "expected_delivery": (TODAY - dt.timedelta(days=5)).isoformat(),
                       "delivery_date": (TODAY - dt.timedelta(days=5)).isoformat(),
                       "delivery_status": "delivered", "delivery_confirmation": "photo"},
        # High value, and the photo turns out to be one already on file.
        "ORD-100128": {"customer_id": "CUST-0031", "category": "electronics",
                       "item_name": "wireless headphones", "value_usd": "899.00", "quantity": 1,
                       "order_date": (TODAY - dt.timedelta(days=7)).isoformat(),
                       "expected_delivery": (TODAY - dt.timedelta(days=3)).isoformat(),
                       "delivery_date": (TODAY - dt.timedelta(days=3)).isoformat(),
                       "delivery_status": "delivered", "delivery_confirmation": "signature"},
        # Mid value, late in the window, and the account given does not match the record.
        "ORD-100077": {"customer_id": "CUST-0019", "category": "electronics",
                       "item_name": "portable monitor", "value_usd": "320.00", "quantity": 1,
                       "order_date": (TODAY - dt.timedelta(days=16)).isoformat(),
                       "expected_delivery": (TODAY - dt.timedelta(days=12)).isoformat(),
                       "delivery_date": (TODAY - dt.timedelta(days=12)).isoformat(),
                       "delivery_status": "delivered", "delivery_confirmation": "photo"},
        # A strong not-received packet: full carrier trail and matching address.
        "ORD-100155": {"customer_id": "CUST-0012", "category": "books",
                       "item_name": "field guide", "value_usd": "68.50", "quantity": 1,
                       "order_date": (TODAY - dt.timedelta(days=30)).isoformat(),
                       "expected_delivery": (TODAY - dt.timedelta(days=25)).isoformat(),
                       "delivery_date": (TODAY - dt.timedelta(days=25)).isoformat(),
                       "delivery_status": "delivered", "delivery_confirmation": "signature",
                       "billing_matches_shipping": "yes", "avs_result": "full_match",
                       "cvv_result": "match", "three_ds": "authenticated",
                       "ip_match_prior_orders": "yes", "device_match_prior_orders": "yes"},
        # A weak fraud packet: nothing corroborates the cardholder being the buyer.
        "ORD-100190": {"customer_id": "CUST-0044", "category": "electronics",
                       "item_name": "action camera", "value_usd": "512.00", "quantity": 1,
                       "order_date": (TODAY - dt.timedelta(days=40)).isoformat(),
                       "expected_delivery": (TODAY - dt.timedelta(days=35)).isoformat(),
                       "delivery_date": "", "delivery_status": "in_transit",
                       "delivery_confirmation": "", "billing_matches_shipping": "no",
                       "avs_result": "no_match", "cvv_result": "not_provided",
                       "three_ds": "not_used", "ip_match_prior_orders": "no",
                       "device_match_prior_orders": "no"},
    }
    by_id = {row["order_id"]: row for row in rows}
    for order_id, changes in pinned.items():
        by_id[order_id].update(changes)

    fields = list(rows[0])
    with (SAMPLE_DIR / "orders.csv").open("w", encoding="utf-8", newline="") as handle:
        handle.write(f"# {SYNTHETIC_NOTE}\n")
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    return rows


def _write_photos_and_hashes() -> None:
    """Write the claim photos, and the hash file the reuse check compares against.

    One photo is deliberately the same picture filed under an earlier claim by a
    different customer, which is the signal the high-risk scenario turns on.
    """
    photo_dir = SAMPLE_DIR / "photos"
    photo_dir.mkdir(parents=True, exist_ok=True)

    delivered_042 = dt.datetime.combine(TODAY - dt.timedelta(days=5), dt.time(14, 30))
    delivered_128 = dt.datetime.combine(TODAY - dt.timedelta(days=3), dt.time(9, 10))

    # Scenario one: ordinary photos taken after delivery, with capture metadata.
    _save_photo(_photo(11, "item"), photo_dir / "claim_042_item.png",
                delivered_042 + dt.timedelta(hours=6))
    _save_photo(_photo(12, "packaging"), photo_dir / "claim_042_packaging.png",
                delivered_042 + dt.timedelta(hours=6, minutes=4))

    # An older claim from a different customer. Its picture is the one that reappears.
    reused = _photo(77, "item")
    _save_photo(reused, photo_dir / "prior_claim_ch0091.png",
                dt.datetime(2026, 1, 8, 11, 0))

    # Scenario two: the same picture again, re-saved, and a second photo whose
    # capture time predates delivery.
    _save_photo(reused, photo_dir / "claim_128_item.png", dt.datetime(2026, 1, 8, 11, 0))
    _save_photo(_photo(13, "packaging"), photo_dir / "claim_128_packaging.png",
                delivered_128 - dt.timedelta(days=2))

    # Scenario three: a photo stripped of metadata, as a messaging app would leave it.
    _save_photo(_photo(21, "screen"), photo_dir / "claim_077_screen.png", None)

    # A second unrelated prior image, so the file is not a single entry. It must be
    # a picture no current scenario supplies: seeding it with one of them would make
    # the clean scenario report a reuse match against itself.
    _save_photo(_photo(88, "item"), photo_dir / "prior_claim_ch0104.png",
                dt.datetime(2026, 2, 2, 15, 30))

    prior = [
        {"image_id": "IMG-CH0091", "claim_id": "CLM-2026-0091", "customer_id": "CUST-0052",
         "filed_on": "2026-01-08", "phash": image_phash(photo_dir / "prior_claim_ch0091.png")},
        {"image_id": "IMG-CH0104", "claim_id": "CLM-2026-0104", "customer_id": "CUST-0018",
         "filed_on": "2026-02-02", "phash": image_phash(photo_dir / "prior_claim_ch0104.png")},
    ]
    with (SAMPLE_DIR / "claim_image_hashes.csv").open("w", encoding="utf-8", newline="") as handle:
        handle.write(f"# {SYNTHETIC_NOTE}\n")
        writer = csv.DictWriter(handle, fieldnames=list(prior[0]))
        writer.writeheader()
        writer.writerows(prior)


def _write_claims() -> None:
    claims = {
        "return_simple_low": {
            "order_id": "ORD-100042", "claim_type": "damaged",
            "claim_text": "One sleeve arrived with a tear near the cuff. Photos attached.",
            "chat_transcript": (
                "Customer: Hello, the overshirt arrived with a tear on the left sleeve.\n"
                "Agent: I am sorry about that. Could you send a photo of the item and the "
                "packaging?\n"
                "Customer: Just sent both. Happy with a replacement if that is easier.\n"
                "Agent: Thank you, I will pass this to the returns team."
            ),
            "photos": ["photos/claim_042_item.png", "photos/claim_042_packaging.png"],
        },
        "return_high_value_reused_photo": {
            "order_id": "ORD-100128", "claim_type": "damaged",
            "claim_text": "Headphones were crushed in the box. I need a full refund now.",
            "chat_transcript": (
                "Customer: The headphones are destroyed. I want a full refund immediately.\n"
                "Agent: I am sorry. Could you send photographs of the item and the box?\n"
                "Customer: Sent. This is urgent, I need my money back today or I will "
                "escalate to my bank and start a chargeback.\n"
                "Agent: Understood, I am raising this for review.\n"
                "Customer: Get me a manager right now."
            ),
            "photos": ["photos/claim_128_item.png", "photos/claim_128_packaging.png"],
        },
        "return_inconsistent_story": {
            "order_id": "ORD-100077", "claim_type": "not_as_described",
            "claim_text": "The mesh router I ordered is the wrong model and arrived last week.",
            "chat_transcript": (
                "Customer: The mesh router is not the model shown on the listing.\n"
                "Agent: Could I check the order? I have a portable monitor on this order.\n"
                "Customer: It arrived on the 2nd of April and it is the wrong one.\n"
                "Agent: The record shows delivery on a different date. Could you send a "
                "photo of the item you received?\n"
                "Customer: I do not have one to hand. Please escalate this to a supervisor, "
                "this is urgent."
            ),
            "photos": ["photos/claim_077_screen.png"],
        },
    }
    for name, claim in claims.items():
        claim["note"] = SYNTHETIC_NOTE
        (SAMPLE_DIR / f"{name}.json").write_text(
            json.dumps(claim, indent=2) + "\n", encoding="utf-8")


def _write_disputes() -> None:
    disputes = {
        "dispute_not_received_strong": {
            "dispute_id": "DSP-2026-0007", "order_id": "ORD-100155",
            "reason_category": "not_received",
            "network_reason_code": "sample code 13.1 (illustrative, confirm with your processor)",
            "amount": 68.50, "currency": "USD",
            "notice_date": (TODAY - dt.timedelta(days=4)).isoformat(),
            "response_window_days": 20,
        },
        "dispute_fraud_weak": {
            "dispute_id": "DSP-2026-0011", "order_id": "ORD-100190",
            "reason_category": "fraud_unauthorized",
            "network_reason_code": "",
            "amount": 512.00, "currency": "USD",
            "notice_date": (TODAY - dt.timedelta(days=9)).isoformat(),
            # No window given, so the workflow has to fall back and say it did.
        },
    }
    for name, dispute in disputes.items():
        dispute["note"] = SYNTHETIC_NOTE
        (SAMPLE_DIR / f"{name}.json").write_text(
            json.dumps(dispute, indent=2) + "\n", encoding="utf-8")


def _write_sellers() -> None:
    sellers = [
        {"seller_id": "SEL-2001", "display_name": "Northlight Audio (fictional)",
         "joined": "2021-06-14", "rating": "4.7", "orders_90d": "1840",
         "category": "electronics", "prior_violations": "0", "prior_appeals_upheld": "0",
         "authorized_distributor": "yes"},
        {"seller_id": "SEL-2044", "display_name": "Bright Market Goods (fictional)",
         "joined": "2025-09-02", "rating": "4.1", "orders_90d": "620",
         "category": "electronics", "prior_violations": "2", "prior_appeals_upheld": "0",
         "authorized_distributor": "no"},
    ]
    with (SAMPLE_DIR / "sellers.csv").open("w", encoding="utf-8", newline="") as handle:
        handle.write(f"# {SYNTHETIC_NOTE}\n")
        writer = csv.DictWriter(handle, fieldnames=list(sellers[0]))
        writer.writeheader()
        writer.writerows(sellers)

    events = [
        {"seller_id": "SEL-2001", "date": "2026-03-02", "event": "ip_complaint_received",
         "detail": "Rights owner complaint on one listing."},
        {"seller_id": "SEL-2001", "date": "2026-03-03", "event": "listing_removed",
         "detail": "Listing taken down pending review."},
        {"seller_id": "SEL-2001", "date": "2026-03-05", "event": "appeal_submitted",
         "detail": "Seller supplied distributor invoices."},
        {"seller_id": "SEL-2044", "date": "2026-01-19", "event": "counterfeit_complaint",
         "detail": "Two buyer complaints on one SKU."},
        {"seller_id": "SEL-2044", "date": "2026-02-11", "event": "warning_issued",
         "detail": "Warning on listing accuracy."},
        {"seller_id": "SEL-2044", "date": "2026-03-28", "event": "counterfeit_complaint",
         "detail": "Further complaints on the same SKU."},
        {"seller_id": "SEL-2044", "date": "2026-03-30", "event": "listings_suspended",
         "detail": "Category listings suspended."},
        {"seller_id": "SEL-2044", "date": "2026-04-01", "event": "appeal_submitted",
         "detail": "Seller supplied a supplier invoice."},
    ]
    with (SAMPLE_DIR / "seller_violations.csv").open("w", encoding="utf-8", newline="") as handle:
        handle.write(f"# {SYNTHETIC_NOTE}\n")
        writer = csv.DictWriter(handle, fieldnames=list(events[0]))
        writer.writeheader()
        writer.writerows(events)

    sales = [
        {"seller_id": "SEL-2001", "sku": "NL-HP-300", "units_sold": "180",
         "first_sale_date": "2026-01-20", "last_sale_date": "2026-03-02"},
        {"seller_id": "SEL-2044", "sku": "BM-EB-12", "units_sold": "640",
         "first_sale_date": "2025-12-04", "last_sale_date": "2026-03-29"},
    ]
    with (SAMPLE_DIR / "seller_sales.csv").open("w", encoding="utf-8", newline="") as handle:
        handle.write(f"# {SYNTHETIC_NOTE}\n")
        writer = csv.DictWriter(handle, fieldnames=list(sales[0]))
        writer.writeheader()
        writer.writerows(sales)

    with (SAMPLE_DIR / "authorized_distributors.csv").open(
            "w", encoding="utf-8", newline="") as handle:
        handle.write(f"# {SYNTHETIC_NOTE}\n")
        handle.write("supplier_name,brand,status\n")
        handle.write("Harbour Distribution Ltd (fictional),Northlight Audio,authorized\n")
        handle.write("Meridian Trade Supply (fictional),Bright Market,unknown\n")


# Invoices are stored as text so the extractor has something to parse that is not
# already structured, which is the situation it exists for.
INVOICE_STRONG = """SUPPLIER INVOICE
Supplier: Harbour Distribution Ltd (fictional)
Invoice Number: HD-2026-4471
Invoice Date: 2026-01-06
Bill To: Northlight Audio (fictional)

SKU         Description                 Qty      Unit Price     Line Total
NL-HP-300   Northlight headphones 300   200      41.50          8300.00

Subtotal: 8300.00
Tax: 0.00
Total: 8300.00 USD
"""

INVOICE_MISMATCH = """SUPPLIER INVOICE
Supplier: Meridian Trade Supply (fictional)
Invoice Number: MT-8823
Invoice Date: 2026-02-20
Bill To: Bright Market Goods (fictional)

SKU        Description              Qty     Unit Price     Line Total
BM-EB-12   Earbuds model 12         120     6.20           744.00

Subtotal: 744.00
Tax: 0.00
Total: 744.00 USD
"""

APPEAL_STRONG = """Appeal regarding the removal of listing NL-HP-300.

Root cause: we listed stock from a new distributor without attaching the
authorisation letter to the listing, so the rights owner had no way to verify our
source and filed a complaint.

Corrective action: we have attached the distributor invoice HD-2026-4471 from
Harbour Distribution Ltd and the brand authorisation letter to the listing record.

Preventive measures: from this month no listing in this brand goes live until the
authorisation document is attached, and we have added a weekly check on the four
listings in this category.

Evidence: invoice HD-2026-4471 dated 6 January 2026 for 200 units, against 180
units sold since 20 January 2026.

We would be grateful for reinstatement once you have reviewed the documents.
"""

APPEAL_WEAK = """Please reinstate our listings. We have done nothing wrong and the
complaints are from competitors trying to damage our business. We have sold these
earbuds for months with no issue and our rating is good.

We attach an invoice showing we buy genuine stock.

Please restore the listings as soon as possible as this is costing us money every
day.
"""


def _write_appeals() -> None:
    invoice_dir = SAMPLE_DIR / "invoices"
    invoice_dir.mkdir(parents=True, exist_ok=True)
    (invoice_dir / "invoice_strong.txt").write_text(INVOICE_STRONG, encoding="utf-8")
    (invoice_dir / "invoice_mismatch.txt").write_text(INVOICE_MISMATCH, encoding="utf-8")

    appeals = {
        "appeal_strong_authorized_reseller": {
            "seller_id": "SEL-2001", "violation_type": "ip_complaint",
            "sku": "NL-HP-300",
            "appeal_letter": APPEAL_STRONG,
            "documents": ["invoices/invoice_strong.txt"],
        },
        "appeal_invoice_mismatch": {
            "seller_id": "SEL-2044", "violation_type": "counterfeit",
            "sku": "BM-EB-12",
            "appeal_letter": APPEAL_WEAK,
            "documents": ["invoices/invoice_mismatch.txt"],
        },
    }
    for name, appeal in appeals.items():
        appeal["note"] = SYNTHETIC_NOTE
        (SAMPLE_DIR / f"{name}.json").write_text(
            json.dumps(appeal, indent=2) + "\n", encoding="utf-8")


def ensure_samples(force: bool = False) -> None:
    """Write the bundled e-commerce samples if they are missing or out of date."""
    SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
    stamp = SAMPLE_DIR / ".generator_version"
    current = stamp.read_text(encoding="utf-8").strip() if stamp.is_file() else ""
    if not force and current == str(SAMPLE_GENERATOR_VERSION) and \
            (SAMPLE_DIR / "orders.csv").is_file():
        return

    customers = _write_customers()
    _write_orders(customers)
    _write_photos_and_hashes()
    _write_claims()
    _write_disputes()
    _write_sellers()
    _write_appeals()
    stamp.write_text(str(SAMPLE_GENERATOR_VERSION) + "\n", encoding="utf-8")

## 3. Return dispute tools

Order and account lookup, the policy window check,
image and conversation signals, the weighted score, and routing.

In [ ]:
class OrderLookupArgs(BaseModel):
    order_id: str = Field(default="", description="Order reference.")
    sample_name: str = Field(default="", description="Bundled claim scenario to read instead.")


class CustomerHistoryArgs(BaseModel):
    customer_id: str = Field(default="", description="Customer reference.")
    sample_name: str = Field(default="", description="Take the customer from a bundled claim.")


class PolicyCheckArgs(BaseModel):
    order_id: str = Field(default="")
    claim_type: str = Field(default="", description="damaged, not_as_described, missing_item, "
                                                    "not_received or changed_mind.")
    claim_date: str = Field(default="", description="ISO date; today's sample date if empty.")
    evidence_supplied: list[str] = Field(default_factory=list)
    sample_name: str = Field(default="")


class ImageSignalsArgs(BaseModel):
    photo_paths: list[str] = Field(default_factory=list, description="Paths to supplied photos.")
    customer_id: str = Field(default="")
    delivery_date: str = Field(default="", description="ISO date the order was delivered.")
    sample_name: str = Field(default="")


class TranscriptSignalsArgs(BaseModel):
    text: str = Field(default="", description="The conversation with the customer.")
    order_id: str = Field(default="", description="Checked against the order record.")
    sample_name: str = Field(default="")


class RiskScoreArgs(BaseModel):
    features: dict[str, Any] = Field(default_factory=dict,
                                     description="Observations from the other tools.")
    sample_name: str = Field(
        default="", description="Gather the observations from a bundled claim.")


class RouteClaimArgs(BaseModel):
    order_value_usd: float = Field(default=-1.0, description="Order value; measured if absent.")
    score: float = Field(default=-1.0, description="Risk score 0-100; measured if absent.")
    sample_name: str = Field(default="")


def _read_rows(name: str) -> list[dict[str, str]]:
    ensure_samples()
    path = SAMPLE_DIR / name
    lines = [line for line in path.read_text(encoding="utf-8").splitlines()
             if not line.startswith("#")]
    return list(csv.DictReader(lines))


def _load_claim(sample_name: str) -> dict[str, Any]:
    ensure_samples()
    stem = (sample_name or "return_simple_low").removesuffix(".json")
    path = SAMPLE_DIR / f"{stem}.json"
    if not path.is_file():
        raise FileNotFoundError(f"no bundled claim named '{stem}'")
    return json.loads(path.read_text(encoding="utf-8"))


def _order_row(order_id: str) -> dict[str, str] | None:
    return next((r for r in _read_rows("orders.csv") if r["order_id"] == order_id), None)


def _customer_row(customer_id: str) -> dict[str, str] | None:
    return next((r for r in _read_rows("customers.csv") if r["customer_id"] == customer_id), None)


def _date(value: str) -> dt.date | None:
    try:
        return dt.date.fromisoformat(value)
    except (TypeError, ValueError):
        return None


@tool(args_schema=OrderLookupArgs)
def order_lookup(order_id: str = "", sample_name: str = "") -> dict[str, Any]:
    """Look up an order: items, value, dates, delivery status and the customer reference.

    The shipping address and card number are masked here rather than later, so the
    unmasked forms never reach a step result or a brief.
    """
    try:
        reference = order_id or _load_claim(sample_name)["order_id"]
    except (FileNotFoundError, KeyError, json.JSONDecodeError) as exc:
        return {"error": str(exc), "tool_version": 1}

    row = _order_row(reference)
    if row is None:
        return {"error": f"no order '{reference}' in the sample set", "tool_version": 1}

    return {
        "order_id": row["order_id"], "customer_id": row["customer_id"],
        "order_date": row["order_date"], "expected_delivery": row["expected_delivery"],
        "delivery_date": row["delivery_date"], "delivery_status": row["delivery_status"],
        "category": row["category"], "item_name": row["item_name"],
        "quantity": int(row["quantity"]), "value_usd": float(row["value_usd"]),
        "currency": row["currency"], "carrier": row["carrier"],
        "tracking_number": row["tracking_number"],
        "shipping_address": mask_address(row["shipping_address"]),
        "card_last4": mask_card(row["card_last4"]),
        "delivery_confirmation": row["delivery_confirmation"],
        "high_value": bool(float(row["value_usd"]) >= HIGH_VALUE_USD),
        "high_value_threshold_usd": HIGH_VALUE_USD,
        "data_note": SYNTHETIC_NOTE,
        "tool_version": 1,
    }


@tool(args_schema=CustomerHistoryArgs)
def customer_history(customer_id: str = "", sample_name: str = "") -> dict[str, Any]:
    """Account age, order and return counts, and how prior disputes were decided.

    Contact details come back masked. A return rate is a description of an account,
    not a judgement about the person holding it.
    """
    reference = customer_id
    if not reference:
        try:
            order = _order_row(_load_claim(sample_name)["order_id"])
            reference = order["customer_id"] if order else ""
        except (FileNotFoundError, KeyError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}

    row = _customer_row(reference)
    if row is None:
        return {"error": f"no customer '{reference}' in the sample set", "tool_version": 1}

    opened = _date(row["account_opened"])
    age_days = (TODAY - opened).days if opened else None
    orders = int(row["orders_count"])
    returns = int(row["returns_count"])
    return {
        **masked_customer(row),
        "account_age_days": age_days,
        "orders_count": orders, "returns_count": returns,
        "return_rate": round(returns / orders, 4) if orders else 0.0,
        "prior_disputes": int(row["prior_disputes"]),
        "prior_disputes_upheld_against_customer": int(row["prior_disputes_upheld"]),
        "note": ("A return rate describes an account, not a person. Several categories "
                 "have high return rates for ordinary reasons."),
        "data_note": SYNTHETIC_NOTE,
        "tool_version": 1,
    }


@tool(args_schema=PolicyCheckArgs)
def policy_check(order_id: str = "", claim_type: str = "", claim_date: str = "",
                 evidence_supplied: list[str] | None = None,
                 sample_name: str = "") -> dict[str, Any]:
    """Compare a claim against the return window, category rules and evidence needed.

    Reports the window it used and the date it measured from, so a reader can check
    the arithmetic rather than take the verdict.
    """
    try:
        claim = _load_claim(sample_name) if sample_name else {}
    except (FileNotFoundError, json.JSONDecodeError) as exc:
        return {"error": str(exc), "tool_version": 1}

    reference = order_id or claim.get("order_id", "")
    kind = (claim_type or claim.get("claim_type", "")).lower()
    supplied = list(evidence_supplied or [])
    if not supplied and claim:
        # A photo of the item and of the packaging are inferred from what was sent.
        for path in claim.get("photos", []):
            if "packaging" in path:
                supplied.append("photo_of_packaging")
            elif "item" in path:
                supplied.append("photo_of_item")

    row = _order_row(reference)
    if row is None:
        return {"error": f"no order '{reference}' in the sample set", "tool_version": 1}

    policy = load_return_policy()
    if kind not in policy["evidence_requirements"]:
        return {"error": f"unknown claim type '{kind}'",
                "known": sorted(policy["evidence_requirements"]), "tool_version": 1}

    category = row["category"]
    window = int(policy["category_windows"].get(category, policy["standard_window_days"]))
    measured_from = (row["delivery_date"] or row["expected_delivery"]) \
        if kind in policy["window_from_delivery"] else row["order_date"]
    start = _date(measured_from)
    raised = _date(claim_date) or TODAY
    days_elapsed = (raised - start).days if start else None

    problems: list[str] = []
    within = None
    if days_elapsed is not None:
        within = days_elapsed <= window
        if not within:
            problems.append(f"claim raised {days_elapsed} days after {measured_from}, "
                            f"beyond the {window} day window for {category}")

    excluded = policy["excluded_categories"].get(category)
    if excluded and kind not in excluded["allows_claim_types"]:
        problems.append(f"{excluded['reason']} ({kind} claims are not accepted for {category})")

    requirements = policy["evidence_requirements"][kind]
    missing = [item for item in requirements.get("required", []) if item not in supplied]
    if missing:
        problems.append(f"required evidence not supplied: {', '.join(missing)}")

    too_early = False
    if kind == "not_received":
        expected = _date(row["expected_delivery"])
        minimum = int(policy["minimum_days_before_not_received_claim"])
        if expected and (raised - expected).days < minimum:
            too_early = True
            problems.append(f"raised before {minimum} days past the expected delivery date, "
                            f"which is early to assess rather than a policy failure")

    return {
        "order_id": reference, "claim_type": kind, "category": category,
        "window_days": window, "window_measured_from": measured_from,
        "days_elapsed": days_elapsed, "within_window": within,
        "days_from_window_edge": (window - days_elapsed) if days_elapsed is not None else None,
        "category_excluded": bool(excluded),
        "required_evidence": requirements.get("required", []),
        "helpful_evidence": requirements.get("helpful", []),
        "evidence_supplied": supplied, "missing_required_evidence": missing,
        "raised_too_early": too_early,
        "policy_problems": problems, "meets_policy": not problems,
        "tool_version": 1,
    }


@tool(args_schema=ImageSignalsArgs)
def image_signals(photo_paths: list[str] | None = None, customer_id: str = "",
                  delivery_date: str = "", sample_name: str = "") -> dict[str, Any]:
    """Describe supplied photos: metadata, dimensions, and reuse against earlier claims.

    These are signals, not proof. Nothing here detects a generated image or shows
    that a photograph is dishonest; a match means two files are the same picture,
    which has ordinary explanations a reviewer should check first.
    """
    try:
        claim = _load_claim(sample_name) if sample_name else {}
    except (FileNotFoundError, json.JSONDecodeError) as exc:
        return {"error": str(exc), "tool_version": 1}

    paths = list(photo_paths or [])
    if not paths and claim:
        paths = [str(SAMPLE_DIR / p) for p in claim.get("photos", [])]
    if not paths:
        return {"error": "no photos supplied", "tool_version": 1}

    order = _order_row(claim.get("order_id", "")) if claim else None
    delivered = _date(delivery_date) or (_date(order["delivery_date"]) if order else None)
    reference_customer = customer_id or (order["customer_id"] if order else "")

    from PIL import Image

    prior = _read_rows("claim_image_hashes.csv")
    results, matches = [], []
    for raw in paths:
        path = pathlib.Path(raw)
        if not path.is_file():
            results.append({"file": path.name, "error": "file not found"})
            continue
        try:
            with Image.open(path) as handle:
                width, height = handle.size
                exif = dict(handle.getexif() or {})
                digest = image_phash(path)
        except Exception as exc:  # noqa: BLE001 - an unreadable upload is a finding
            results.append({"file": path.name, "error": f"could not read: {type(exc).__name__}"})
            continue

        captured_raw = exif.get(0x0132, "")
        captured = None
        if captured_raw:
            try:
                captured = dt.datetime.strptime(captured_raw, "%Y:%m:%d %H:%M:%S").date()
            except ValueError:
                captured = None

        nearest = None
        for row in prior:
            distance = hamming(digest, row["phash"])
            if nearest is None or distance < nearest["distance"]:
                nearest = {"distance": distance, **row}
        matched = bool(nearest and nearest["distance"] <= IMAGE_MATCH_DISTANCE)
        other_customer = bool(matched and nearest["customer_id"] != reference_customer)

        entry = {
            "file": path.name, "width": width, "height": height,
            "bytes": path.stat().st_size,
            "has_capture_metadata": bool(captured_raw),
            "capture_date": captured.isoformat() if captured else "",
            "software_tag": exif.get(0x0131, ""),
            "captured_before_delivery": bool(captured and delivered and captured < delivered),
            "nearest_prior_claim": (
                {"claim_id": nearest["claim_id"], "customer_id": nearest["customer_id"],
                 "filed_on": nearest["filed_on"], "distance": nearest["distance"]}
                if nearest else None),
            "matches_prior_claim": matched,
            "match_is_another_customer": other_customer,
        }
        results.append(entry)
        if matched:
            matches.append(entry)

    readable = [r for r in results if "error" not in r]
    return {
        "photos": results, "photo_count": len(results),
        "photos_missing_metadata": sum(1 for r in readable if not r["has_capture_metadata"]),
        "photos_captured_before_delivery": sum(1 for r in readable
                                               if r["captured_before_delivery"]),
        "reuse_matches": len(matches),
        "reuse_against_other_customer": sum(1 for r in matches if r["match_is_another_customer"]),
        "match_distance_threshold": IMAGE_MATCH_DISTANCE,
        "delivery_date_compared": delivered.isoformat() if delivered else "",
        "interpretation_note": (
            "Signals, not proof. Absent metadata is normal because messaging apps strip "
            "it. A reuse match means two files are the same picture, which can happen "
            "through a shared account or a re-sent photo, and says nothing on its own "
            "about intent. Nothing here identifies a generated image."),
        "tool_version": 1,
    }


@tool(args_schema=TranscriptSignalsArgs)
def transcript_signals(text: str = "", order_id: str = "",
                       sample_name: str = "") -> dict[str, Any]:
    """Check a conversation against the order record and note escalation phrasing.

    Inconsistencies are reported with both values so a reviewer can see what
    disagreed. Phrasing counts are a prompt to read the conversation, nothing more.
    """
    try:
        claim = _load_claim(sample_name) if sample_name else {}
    except (FileNotFoundError, json.JSONDecodeError) as exc:
        return {"error": str(exc), "tool_version": 1}

    body = text or claim.get("chat_transcript", "")
    if not body:
        return {"error": "no transcript supplied", "tool_version": 1}
    reference = order_id or claim.get("order_id", "")
    row = _order_row(reference)

    lowered = body.lower()
    pressure = sorted({p for p in PRESSURE_PHRASES if p in lowered})
    demands = sorted({p for p in REFUND_DEMAND_PHRASES if p in lowered})

    inconsistencies = []
    if row is not None:
        item = row["item_name"].lower()
        # An item named in the conversation that is not the item on the order.
        others = {name.lower() for names in ITEM_NAMES.values() for name in names} - {item}
        mentioned = sorted({name for name in others if name in lowered})
        if mentioned and item not in lowered:
            inconsistencies.append({
                "field": "item", "on_order": row["item_name"],
                "in_conversation": mentioned[0],
                "note": "The item described does not match the item on this order."})

        stated_dates = re.findall(
            r"\b(\d{1,2})(?:st|nd|rd|th)?\s+of\s+([A-Z][a-z]+)|\b([A-Z][a-z]+)\s+(\d{1,2})\b", body)
        months = {"january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
                  "july": 7, "august": 8, "september": 9, "october": 10, "november": 11,
                  "december": 12}
        delivered = _date(row["delivery_date"])
        for day_a, month_a, month_b, day_b in stated_dates:
            day, month_name = (day_a, month_a) if day_a else (day_b, month_b)
            month = months.get((month_name or "").lower())
            if not (day and month and delivered):
                continue
            try:
                stated = dt.date(delivered.year, month, int(day))
            except ValueError:
                continue
            if stated != delivered:
                inconsistencies.append({
                    "field": "delivery_date", "on_order": delivered.isoformat(),
                    "in_conversation": stated.isoformat(),
                    "note": "The date given does not match the delivery record."})
                break

    return {
        "order_id": reference, "characters": len(body),
        "inconsistencies": inconsistencies, "inconsistency_count": len(inconsistencies),
        "pressure_phrases": pressure, "pressure_phrase_count": len(pressure),
        "refund_demand_phrases": demands,
        "escalation_mentioned": bool({"escalate", "manager", "supervisor", "lawyer",
                                      "chargeback"} & set(pressure)),
        "interpretation_note": (
            "Urgency is a normal reaction to a genuine failure and is not a risk "
            "finding on its own. An inconsistency may be ordinary misremembering; it "
            "is a question to ask, not a conclusion."),
        "tool_version": 1,
    }


def _features_from_sample(sample_name: str) -> dict[str, Any]:
    """Run the observation tools over a bundled claim and collect what they saw."""
    claim = _load_claim(sample_name)
    order = order_lookup.invoke({"sample_name": sample_name})
    history = customer_history.invoke({"sample_name": sample_name})
    policy = policy_check.invoke({"sample_name": sample_name})
    images = image_signals.invoke({"sample_name": sample_name})
    transcript = transcript_signals.invoke({"sample_name": sample_name})
    return {
        "order_value_usd": order.get("value_usd", 0.0),
        "account_age_days": history.get("account_age_days"),
        "return_rate": history.get("return_rate", 0.0),
        "orders_count": history.get("orders_count", 0),
        "prior_disputes_upheld": history.get("prior_disputes_upheld_against_customer", 0),
        "days_from_window_edge": policy.get("days_from_window_edge"),
        "missing_required_evidence": len(policy.get("missing_required_evidence", [])),
        "image_reuse_other_customer": images.get("reuse_against_other_customer", 0),
        "photos_missing_metadata": images.get("photos_missing_metadata", 0),
        "photos_captured_before_delivery": images.get("photos_captured_before_delivery", 0),
        "transcript_inconsistencies": transcript.get("inconsistency_count", 0),
        "pressure_phrase_count": transcript.get("pressure_phrase_count", 0),
        "repeat_claim_same_category": 0,
        "claim_type": claim.get("claim_type", ""),
    }


@tool(args_schema=RiskScoreArgs)
def risk_score(features: dict[str, Any] | None = None, sample_name: str = "") -> dict[str, Any]:
    """Score a claim 0-100 from the documented weights, listing every factor that fired.

    Anything not supplied is observed from the claim rather than assumed, so the
    score always rests on what the other tools actually saw. Each contributing
    factor is returned with its weight and the ordinary explanation it might have.
    """
    observed = dict(features or {})
    if not observed and sample_name:
        try:
            observed = _features_from_sample(sample_name)
        except (FileNotFoundError, KeyError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}
    if not observed:
        return {"error": "no observations supplied and no sample named", "tool_version": 1}

    rules = load_risk_weights()
    factors = rules["factors"]
    fired: list[dict[str, Any]] = []

    def fire(key: str, observation: str) -> None:
        spec = factors[key]
        fired.append({"factor": key, "weight": spec["weight"], "observation": observation,
                      "innocent_explanation": " ".join(spec["innocent_explanation"].split())})

    age = observed.get("account_age_days")
    if age is not None and age < factors["new_account"]["threshold_days"]:
        fire("new_account", f"account is {age} days old")

    orders = observed.get("orders_count", 0)
    rate = observed.get("return_rate", 0.0)
    if (orders >= factors["high_return_rate"]["minimum_orders"]
            and rate > factors["high_return_rate"]["threshold_rate"]):
        fire("high_return_rate", f"{rate:.0%} of {orders} orders returned")

    upheld = observed.get("prior_disputes_upheld", 0)
    if upheld > factors["prior_disputes_upheld"]["threshold_count"]:
        fire("prior_disputes_upheld", f"{upheld} prior disputes decided against the account")

    value = observed.get("order_value_usd", 0.0)
    if value >= HIGH_VALUE_USD:
        fire("high_value_claim", f"order value {value:.2f} USD is at or above "
                                 f"{HIGH_VALUE_USD:.0f}")

    edge = observed.get("days_from_window_edge")
    if edge is not None and 0 <= edge <= factors["claim_near_window_edge"]["days_from_edge"]:
        fire("claim_near_window_edge", f"{edge} days left in the return window")

    if observed.get("repeat_claim_same_category", 0):
        fire("repeat_claim_same_category", "an earlier claim of this type in this category")

    if observed.get("image_reuse_other_customer", 0):
        fire("image_hash_match_other_claim",
             f"{observed['image_reuse_other_customer']} photo(s) already filed under "
             f"another customer's claim")

    if observed.get("photos_missing_metadata", 0):
        fire("image_missing_metadata",
             f"{observed['photos_missing_metadata']} photo(s) carry no capture metadata")

    if observed.get("photos_captured_before_delivery", 0):
        fire("image_captured_before_delivery",
             f"{observed['photos_captured_before_delivery']} photo(s) dated before delivery")

    if observed.get("missing_required_evidence", 0):
        fire("required_evidence_missing",
             f"{observed['missing_required_evidence']} required item(s) not supplied")

    if observed.get("transcript_inconsistencies", 0):
        fire("transcript_inconsistency",
             f"{observed['transcript_inconsistencies']} inconsistency with the order record")

    if observed.get("pressure_phrase_count", 0) >= 3:
        fire("transcript_pressure_language",
             f"{observed['pressure_phrase_count']} urgency or escalation phrases")

    total = min(sum(f["weight"] for f in fired), int(rules["max_score"]))
    return {
        "score": total, "max_score": int(rules["max_score"]),
        "factors": fired, "factor_count": len(fired),
        "bands": {"high_risk_at_or_above": RISK_HIGH, "specialist_at_or_above": RISK_MEDIUM},
        "disclaimer": " ".join(rules["disclaimer"].split()),
        "observations_used": observed,
        "tool_version": 1,
    }


@tool(args_schema=RouteClaimArgs)
def route_claim(order_value_usd: float = -1.0, score: float = -1.0,
                sample_name: str = "") -> dict[str, Any]:
    """Decide who reviews the claim, by fixed rule from the value and the score.

    This routes the claim; it never decides it. Every level ends with a person
    choosing what to do about the refund.
    """
    value, score = float(order_value_usd), float(score)
    measured: list[str] = []
    if (value < 0 or score < 0) and sample_name:
        try:
            if value < 0:
                value = float(order_lookup.invoke(
                    {"sample_name": sample_name}).get("value_usd", 0.0))
                measured.append("order value")
            if score < 0:
                score = float(risk_score.invoke({"sample_name": sample_name}).get("score", 0.0))
                measured.append("risk score")
        except (FileNotFoundError, KeyError, ValueError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}
    if value < 0 or score < 0:
        return {"error": "need an order value and a risk score, or a sample to take them from",
                "tool_version": 1}

    reasons = []
    if score >= RISK_HIGH:
        level = "HIGH_RISK_REVIEW"
        reasons.append(f"score {score:.0f} is at or above the {RISK_HIGH:.0f} high-risk band")
    elif value < HIGH_VALUE_USD and score < RISK_MEDIUM:
        level = "FAST_TRACK_ELIGIBLE"
        reasons.append(f"value {value:.2f} USD is below {HIGH_VALUE_USD:.0f} and score "
                       f"{score:.0f} is below {RISK_MEDIUM:.0f}")
    else:
        level = "SPECIALIST_REVIEW"
        if value >= HIGH_VALUE_USD:
            reasons.append(f"value {value:.2f} USD is at or above {HIGH_VALUE_USD:.0f}")
        if score >= RISK_MEDIUM:
            reasons.append(f"score {score:.0f} is at or above {RISK_MEDIUM:.0f}")

    return {
        "level": level, "reasons": reasons,
        "order_value_usd": round(value, 2), "risk_score": round(score, 1),
        "measured_here": measured,
        "thresholds": {"high_value_usd": HIGH_VALUE_USD, "risk_high": RISK_HIGH,
                       "risk_medium": RISK_MEDIUM},
        "note": ("Routing only. This decides who reviews the claim, never whether the "
                 "customer is refunded."),
        "tool_version": 1,
    }

## 4. Chargeback tools

The evidence checklist, what the order record holds,
completeness by weight, the likelihood band, the deadline, and a draft
rebuttal citing exhibits.

In [ ]:
class ChecklistArgs(BaseModel):
    reason_category: str = Field(default="", description="The dispute reason category.")
    sample_name: str = Field(default="", description="Bundled dispute to read instead.")


class GatherEvidenceArgs(BaseModel):
    order_id: str = Field(default="")
    sample_name: str = Field(default="")


class CompletenessArgs(BaseModel):
    reason_category: str = Field(default="")
    evidence: dict[str, Any] = Field(default_factory=dict,
                                     description="What gather_evidence found.")
    sample_name: str = Field(default="")


class LikelihoodArgs(BaseModel):
    reason_category: str = Field(default="")
    completeness: float = Field(default=-1.0, description="Share of available weight, 0 to 1.")
    signals: dict[str, Any] = Field(default_factory=dict)
    sample_name: str = Field(default="")


class DeadlineArgs(BaseModel):
    notice_date: str = Field(default="", description="ISO date the notice was received.")
    response_window_days: int = Field(default=0, description="Zero means use the default.")
    sample_name: str = Field(default="")


class RebuttalArgs(BaseModel):
    reason_category: str = Field(default="")
    evidence: dict[str, Any] = Field(default_factory=dict)
    sample_name: str = Field(default="")


def _load_dispute(sample_name: str) -> dict[str, Any]:
    ensure_samples()
    stem = (sample_name or "dispute_not_received_strong").removesuffix(".json")
    path = SAMPLE_DIR / f"{stem}.json"
    if not path.is_file():
        raise FileNotFoundError(f"no bundled dispute named '{stem}'")
    return json.loads(path.read_text(encoding="utf-8"))


@tool(args_schema=ChecklistArgs)
def evidence_checklist(reason_category: str = "", sample_name: str = "") -> dict[str, Any]:
    """List the evidence a representment packet for this reason category is built from.

    Generic categories only. What a given network accepts, and by when, differs by
    network and processor, so the processor is the authority on both.
    """
    try:
        category = (reason_category or _load_dispute(sample_name)["reason_category"]).lower()
    except (FileNotFoundError, KeyError, json.JSONDecodeError) as exc:
        return {"error": str(exc), "tool_version": 1}

    rules = load_chargeback_rules()
    spec = rules["categories"].get(category)
    if spec is None:
        return {"error": f"unknown reason category '{category}'",
                "known": sorted(rules["categories"]), "tool_version": 1}

    def items(group: str) -> list[dict[str, Any]]:
        return [{"key": key, "label": item["label"], "weight": item["weight"]}
                for key, item in spec[group].items()]

    return {
        "reason_category": category, "description": spec["description"],
        "required": items("required"), "helpful": items("helpful"),
        "required_count": len(spec["required"]), "helpful_count": len(spec["helpful"]),
        "total_available_weight": sum(i["weight"] for i in spec["required"].values())
                                  + sum(i["weight"] for i in spec["helpful"].values()),
        "disclaimer": " ".join(rules["disclaimer"].split()),
        "tool_version": 1,
    }


@tool(args_schema=GatherEvidenceArgs)
def gather_evidence(order_id: str = "", sample_name: str = "") -> dict[str, Any]:
    """Collect what the order record holds that a representment packet can use.

    Reports what is present and what is absent by name, so the gaps are visible
    rather than implied. Card and address details come back masked.
    """
    try:
        reference = order_id or _load_dispute(sample_name)["order_id"]
    except (FileNotFoundError, KeyError, json.JSONDecodeError) as exc:
        return {"error": str(exc), "tool_version": 1}

    row = _order_row(reference)
    if row is None:
        return {"error": f"no order '{reference}' in the sample set", "tool_version": 1}

    delivered = row["delivery_status"] == "delivered"
    carrier_events = []
    ordered = _date(row["order_date"])
    if ordered:
        carrier_events.append({"date": ordered.isoformat(), "event": "label created"})
        carrier_events.append({"date": (ordered + dt.timedelta(days=1)).isoformat(),
                               "event": "collected by carrier"})
        if delivered and _date(row["delivery_date"]):
            carrier_events.append({"date": row["delivery_date"], "event": "out for delivery"})
            confirmation = row["delivery_confirmation"] or "no confirmation"
            carrier_events.append({"date": row["delivery_date"],
                                   "event": f"delivered, {confirmation}"})
        else:
            carrier_events.append({"date": (ordered + dt.timedelta(days=3)).isoformat(),
                                   "event": "in transit, no further scan"})

    found = {
        "avs_result": row["avs_result"],
        "cvv_result": row["cvv_result"] if row["cvv_result"] != "not_provided" else "",
        "three_ds_flag": row["three_ds"] if row["three_ds"] != "not_used" else "",
        "device_ip_match": (row["ip_match_prior_orders"] == "yes"
                            and row["device_match_prior_orders"] == "yes"),
        "tracking_number": row["tracking_number"],
        "carrier_events": carrier_events,
        "delivery_confirmation": row["delivery_confirmation"],
        "signature_or_photo": row["delivery_confirmation"] in {"signature", "photo"},
        "shipping_address_match": row["billing_matches_shipping"] == "yes",
        "customer_communications": True,
        "terms_acceptance": row["terms_accepted"] == "yes",
        "prior_order_history": True,
        "refund_history": float(row["refund_issued_usd"]) > 0,
        "listing_snapshot": True,
        "item_specification": True,
        "return_policy_acceptance": row["terms_accepted"] == "yes",
        "photos_before_dispatch": False,
        "transaction_records": True,
        "order_records": True,
        "subscription_record": False,
        "cancellation_policy": False,
        "cancellation_request_log": False,
        "usage_after_cancellation": False,
        "return_receipt": False,
        "refund_policy": True,
    }
    present = {k: v for k, v in found.items() if v not in (False, "", None, [])}
    return {
        "order_id": reference,
        "evidence": found,
        "present_keys": sorted(present),
        "absent_keys": sorted(k for k in found if k not in present),
        "delivery_status": row["delivery_status"],
        "carrier": row["carrier"],
        "shipping_address": mask_address(row["shipping_address"]),
        "card_last4": mask_card(row["card_last4"]),
        "data_note": SYNTHETIC_NOTE,
        "tool_version": 1,
    }


@tool(args_schema=CompletenessArgs)
def completeness_score(reason_category: str = "", evidence: dict[str, Any] | None = None,
                       sample_name: str = "") -> dict[str, Any]:
    """Score how much of the checklist the gathered evidence covers, by weight.

    Anything not supplied is gathered rather than assumed. Missing items are named,
    because the useful output of this step is the list of what to go and find.
    """
    found = dict(evidence or {})
    category = (reason_category or "").lower()
    measured: list[str] = []
    if sample_name and (not found or not category):
        try:
            dispute = _load_dispute(sample_name)
            category = category or dispute["reason_category"].lower()
            if not found:
                gathered = gather_evidence.invoke({"sample_name": sample_name})
                if "error" in gathered:
                    return gathered
                found = gathered["evidence"]
                measured.append("evidence")
        except (FileNotFoundError, KeyError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}

    rules = load_chargeback_rules()
    spec = rules["categories"].get(category)
    if spec is None:
        return {"error": f"unknown reason category '{category}'",
                "known": sorted(rules["categories"]), "tool_version": 1}

    def holds(key: str) -> bool:
        value = found.get(key)
        return value not in (None, False, "", [], {})

    covered, available = 0, 0
    have, missing = [], []
    for group in ("required", "helpful"):
        for key, item in spec[group].items():
            available += item["weight"]
            entry = {"key": key, "label": item["label"], "weight": item["weight"],
                     "group": group}
            if holds(key):
                covered += item["weight"]
                have.append(entry)
            else:
                missing.append(entry)

    share = covered / available if available else 0.0
    missing_required = [m for m in missing if m["group"] == "required"]
    return {
        "reason_category": category,
        "completeness": round(share, 4),
        "covered_weight": covered, "available_weight": available,
        "evidence_present": have, "evidence_missing": missing,
        "missing_required": missing_required,
        "missing_required_count": len(missing_required),
        "required_total": len(spec["required"]),
        "measured_here": measured,
        "tool_version": 1,
    }


@tool(args_schema=LikelihoodArgs)
def win_likelihood_band(reason_category: str = "", completeness: float = -1.0,
                        signals: dict[str, Any] | None = None,
                        sample_name: str = "") -> dict[str, Any]:
    """Place the packet in a band from how complete it is and what the evidence shows.

    A rule-based band, not a prediction and not a probability. Outcomes depend on the
    network, the issuer and the reason code, none of which this can see.
    """
    rules = load_chargeback_rules()
    category = (reason_category or "").lower()
    share = float(completeness)
    extra = dict(signals or {})
    measured: list[str] = []

    if sample_name and (share < 0 or not category):
        try:
            dispute = _load_dispute(sample_name)
            category = category or dispute["reason_category"].lower()
            if share < 0:
                scored = completeness_score.invoke({"sample_name": sample_name})
                if "error" in scored:
                    return scored
                share = float(scored["completeness"])
                extra.setdefault("missing_required_count", scored["missing_required_count"])
                measured.append("completeness")
            if not extra.get("evidence"):
                gathered = gather_evidence.invoke({"sample_name": sample_name})
                extra.setdefault("delivery_confirmed",
                                 bool(gathered.get("evidence", {}).get("delivery_confirmation")))
                extra.setdefault("address_match",
                                 bool(gathered.get("evidence", {}).get("shipping_address_match")))
        except (FileNotFoundError, KeyError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}

    if category not in rules["categories"]:
        return {"error": f"unknown reason category '{category}'",
                "known": sorted(rules["categories"]), "tool_version": 1}
    if share < 0:
        return {"error": "need a completeness share, or a sample to measure one from",
                "tool_version": 1}

    bands = rules["bands"]
    reasons = [f"evidence completeness is {share:.0%} by weight"]
    band = ("HIGH" if share >= bands["strong_at_or_above"]
            else "LOW" if share < bands["weak_below"] else "MEDIUM")

    missing_required = int(extra.get("missing_required_count", 0))
    if missing_required and band == "HIGH":
        band = "MEDIUM"
        reasons.append(f"{missing_required} required item(s) still missing, so the band is "
                       f"held below strong")
    elif missing_required:
        reasons.append(f"{missing_required} required item(s) missing")

    if category == "not_received" and extra.get("delivery_confirmed"):
        reasons.append("delivery is confirmed by the carrier record")
    if category == "fraud_unauthorized" and extra.get("address_match") is False:
        reasons.append("the ship-to address does not match the billing address")

    level = ("STRONG_PACKET" if band == "HIGH"
             else "ACCEPT_LIABILITY_SUGGESTED" if band == "LOW" else "WEAK_PACKET")
    return {
        "reason_category": category, "band": band, "level": level,
        "completeness": round(share, 4), "reasons": reasons,
        "thresholds": {"strong_at_or_above": bands["strong_at_or_above"],
                       "weak_below": bands["weak_below"]},
        "measured_here": measured,
        "note": " ".join(rules["likelihood_note"].split()),
        "tool_version": 1,
    }


@tool(args_schema=DeadlineArgs)
def response_deadline(notice_date: str = "", response_window_days: int = 0,
                      sample_name: str = "") -> dict[str, Any]:
    """Work out when the response is due, and say when the window was assumed.

    A window the notice did not state is a guess from configuration, and is flagged
    as one: missing the real deadline forfeits the dispute whatever the packet holds.
    """
    rules = load_chargeback_rules()
    notice, window = notice_date, int(response_window_days)
    if sample_name and (not notice or window <= 0):
        try:
            dispute = _load_dispute(sample_name)
            notice = notice or dispute.get("notice_date", "")
            window = window or int(dispute.get("response_window_days") or 0)
        except (FileNotFoundError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}

    start = _date(notice)
    if start is None:
        return {"error": f"could not read a notice date from '{notice}'", "tool_version": 1}

    assumed = window <= 0
    if assumed:
        window = int(rules["default_response_window_days"])
    due = start + dt.timedelta(days=window)
    remaining = (due - TODAY).days

    return {
        "notice_date": start.isoformat(), "response_window_days": window,
        "window_was_assumed": assumed,
        "due_date": due.isoformat(),
        "days_remaining": remaining,
        "overdue": remaining < 0,
        "as_of": TODAY.isoformat(),
        "confirm_note": (" ".join(rules["deadline_note"].split()) if assumed
                         else "Window taken from the notice. Confirm with your processor."),
        "tool_version": 1,
    }


@tool(args_schema=RebuttalArgs)
def draft_rebuttal(reason_category: str = "", evidence: dict[str, Any] | None = None,
                   sample_name: str = "") -> dict[str, Any]:
    """Assemble a draft rebuttal letter that cites the exhibits by number.

    A draft for a person to check, edit and submit through their processor. Nothing
    here sends anything to anyone.
    """
    found = dict(evidence or {})
    category = (reason_category or "").lower()
    dispute: dict[str, Any] = {}
    if sample_name:
        try:
            dispute = _load_dispute(sample_name)
            category = category or dispute["reason_category"].lower()
            if not found:
                gathered = gather_evidence.invoke({"sample_name": sample_name})
                if "error" in gathered:
                    return gathered
                found = gathered["evidence"]
        except (FileNotFoundError, KeyError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}

    rules = load_chargeback_rules()
    spec = rules["categories"].get(category)
    if spec is None:
        return {"error": f"unknown reason category '{category}'",
                "known": sorted(rules["categories"]), "tool_version": 1}

    exhibits, index = [], 1
    for group in ("required", "helpful"):
        for key, item in spec[group].items():
            value = found.get(key)
            if value in (None, False, "", [], {}):
                continue
            exhibits.append({"exhibit": f"E{index}", "key": key, "label": item["label"]})
            index += 1

    order_id = dispute.get("order_id", "the order")
    dispute_id = dispute.get("dispute_id", "this dispute")
    amount = dispute.get("amount")
    lines = [
        f"Re: {dispute_id} — representment for order {order_id}",
        "",
        f"We are responding to a dispute raised under {category.replace('_', ' ')}"
        + (f" for {amount:.2f} {dispute.get('currency', 'USD')}." if amount else "."),
        "",
        "The following exhibits are attached:",
    ]
    lines += [f"  {e['exhibit']}. {e['label']}" for e in exhibits] or ["  (no exhibits gathered)"]
    lines += [
        "",
        "Taken together these records address the reason given for the dispute. "
        "We ask that the charge be upheld on the basis of the attached evidence.",
        "",
        "[Draft. Review every statement against the exhibits, confirm the deadline and "
        "the accepted evidence with your processor, and submit through them.]",
    ]

    missing = [{"key": key, "label": item["label"]}
               for key, item in spec["required"].items()
               if found.get(key) in (None, False, "", [], {})]
    return {
        "title": f"Draft representment — {dispute_id}",
        "reason_category": category,
        "body": "\n".join(lines),
        "exhibits": exhibits, "exhibit_count": len(exhibits),
        "missing_required": missing,
        "is_draft": True,
        "note": "A draft for review. This system does not submit anything to any processor.",
        "tool_version": 1,
    }

## 5. Seller appeal tools

Seller standing, the enforcement timeline, invoice
extraction, consistency against the sales record, and appeal structure.

In [ ]:
class SellerProfileArgs(BaseModel):
    seller_id: str = Field(default="")
    sample_name: str = Field(default="", description="Bundled appeal to read instead.")


class TimelineArgs(BaseModel):
    seller_id: str = Field(default="")
    sample_name: str = Field(default="")


class ExtractInvoiceArgs(BaseModel):
    document_paths: list[str] = Field(default_factory=list, description="Invoice text files.")
    sample_name: str = Field(default="")


class InvoiceConsistencyArgs(BaseModel):
    invoices: list[dict[str, Any]] = Field(default_factory=list)
    seller_id: str = Field(default="")
    sample_name: str = Field(default="")


class AppealQualityArgs(BaseModel):
    letter: str = Field(default="")
    sample_name: str = Field(default="")


# What a complete appeal is expected to cover. Matching on wording is crude, so a
# missing section is a question for the reviewer rather than a conclusion.
APPEAL_SECTIONS = {
    "root_cause": ("root cause", "why this happened", "the cause was", "happened because",
                   "we listed", "reason this occurred"),
    "corrective_action": ("corrective action", "we have attached", "we have removed",
                          "we have corrected", "we have updated", "steps we have taken"),
    "preventive_measures": ("preventive", "prevent", "going forward", "from this month",
                            "we have added", "no listing", "weekly check"),
    "evidence_referenced": ("invoice", "authorisation", "authorization", "document",
                            "attached", "evidence"),
}
DEFLECTION_PHRASES = ("competitor", "competitors", "nothing wrong", "not our fault",
                      "we have done nothing", "unfair", "mistake on your side")


def _load_appeal(sample_name: str) -> dict[str, Any]:
    ensure_samples()
    stem = (sample_name or "appeal_strong_authorized_reseller").removesuffix(".json")
    path = SAMPLE_DIR / f"{stem}.json"
    if not path.is_file():
        raise FileNotFoundError(f"no bundled appeal named '{stem}'")
    return json.loads(path.read_text(encoding="utf-8"))


@tool(args_schema=SellerProfileArgs)
def seller_profile(seller_id: str = "", sample_name: str = "") -> dict[str, Any]:
    """Tenure, rating, recent volume, and how prior violations were decided."""
    try:
        reference = seller_id or _load_appeal(sample_name)["seller_id"]
    except (FileNotFoundError, KeyError, json.JSONDecodeError) as exc:
        return {"error": str(exc), "tool_version": 1}

    row = next((r for r in _read_rows("sellers.csv") if r["seller_id"] == reference), None)
    if row is None:
        return {"error": f"no seller '{reference}' in the sample set", "tool_version": 1}

    joined = _date(row["joined"])
    return {
        "seller_id": row["seller_id"], "display_name": row["display_name"],
        "joined": row["joined"],
        "tenure_days": (TODAY - joined).days if joined else None,
        "rating": float(row["rating"]), "orders_90d": int(row["orders_90d"]),
        "category": row["category"],
        "prior_violations": int(row["prior_violations"]),
        "prior_appeals_upheld": int(row["prior_appeals_upheld"]),
        "listed_as_authorized_distributor": row["authorized_distributor"] == "yes",
        "data_note": SYNTHETIC_NOTE,
        "tool_version": 1,
    }


@tool(args_schema=TimelineArgs)
def violation_timeline(seller_id: str = "", sample_name: str = "") -> dict[str, Any]:
    """The complaint, enforcement and listing events for this seller, oldest first."""
    try:
        reference = seller_id or _load_appeal(sample_name)["seller_id"]
    except (FileNotFoundError, KeyError, json.JSONDecodeError) as exc:
        return {"error": str(exc), "tool_version": 1}

    events = [r for r in _read_rows("seller_violations.csv") if r["seller_id"] == reference]
    events.sort(key=lambda r: r["date"])
    if not events:
        return {"seller_id": reference, "events": [], "event_count": 0,
                "note": "No recorded events for this seller.", "tool_version": 1}

    complaints = [e for e in events if "complaint" in e["event"]]
    enforcement = [e for e in events if e["event"] in
                   {"listing_removed", "listings_suspended", "warning_issued"}]
    first, last = _date(events[0]["date"]), _date(events[-1]["date"])
    return {
        "seller_id": reference,
        "events": [{"date": e["date"], "event": e["event"], "detail": e["detail"]}
                   for e in events],
        "event_count": len(events),
        "complaint_count": len(complaints),
        "enforcement_count": len(enforcement),
        "repeat_complaints": len(complaints) > 1,
        "span_days": (last - first).days if first and last else None,
        "tool_version": 1,
    }


@tool(args_schema=ExtractInvoiceArgs)
def extract_invoice_fields(document_paths: list[str] | None = None,
                           sample_name: str = "") -> dict[str, Any]:
    """Pull supplier, dates, line items and totals out of supplied invoice text.

    Parsed with patterns only; the document is read as text and never executed. A
    field that cannot be found is reported missing rather than guessed at.
    """
    paths = list(document_paths or [])
    if not paths and sample_name:
        try:
            appeal = _load_appeal(sample_name)
            paths = [str(SAMPLE_DIR / d) for d in appeal.get("documents", [])]
        except (FileNotFoundError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}
    if not paths:
        return {"error": "no documents supplied", "tool_version": 1}

    invoices = []
    for raw in paths:
        path = pathlib.Path(raw)
        if not path.is_file():
            invoices.append({"file": path.name, "error": "file not found"})
            continue
        text = path.read_text(encoding="utf-8", errors="replace")

        def find(pattern: str, body: str = text) -> str:
            # MULTILINE matters: these patterns anchor on a line start, and without
            # it every header field silently comes back empty. The body is bound as a
            # default rather than read from the loop, so the closure cannot drift onto
            # the next document if this is ever called later than it is now.
            match = re.search(pattern, body, re.I | re.M)
            return match.group(1).strip() if match else ""

        lines = []
        for line in text.splitlines():
            item = re.match(
                r"^\s*([A-Z0-9][A-Z0-9-]{3,})\s+(.+?)\s{2,}(\d+)\s+([\d.]+)\s+([\d.]+)\s*$",
                line)
            if item:
                sku, description, quantity, unit, total = item.groups()
                lines.append({"sku": sku, "description": description.strip(),
                              "quantity": int(quantity), "unit_price": float(unit),
                              "line_total": float(total)})

        missing = []
        supplier = find(r"^Supplier:\s*(.+)$")
        number = find(r"^Invoice Number:\s*(.+)$")
        date_text = find(r"^Invoice Date:\s*(.+)$")
        total_text = find(r"^Total:\s*([\d.]+)")
        for label, value in (("supplier", supplier), ("invoice_number", number),
                             ("invoice_date", date_text), ("total", total_text)):
            if not value:
                missing.append(label)

        invoices.append({
            "file": path.name, "supplier": supplier, "invoice_number": number,
            "invoice_date": date_text, "bill_to": find(r"^Bill To:\s*(.+)$"),
            "line_items": lines, "line_item_count": len(lines),
            "stated_total": float(total_text) if total_text else None,
            "missing_fields": missing,
        })

    return {
        "invoices": invoices, "invoice_count": len(invoices),
        "analysis": "pattern extraction over the document text; nothing is executed",
        "tool_version": 1,
    }


@tool(args_schema=InvoiceConsistencyArgs)
def invoice_consistency(invoices: list[dict[str, Any]] | None = None, seller_id: str = "",
                        sample_name: str = "") -> dict[str, Any]:
    """Check invoiced quantities, dates, suppliers and arithmetic against the sales record.

    Each finding names both numbers so a reviewer can see what disagreed. A shortfall
    is a question to put to the seller, not a conclusion about them.
    """
    supplied = list(invoices or [])
    reference = seller_id
    if sample_name and (not supplied or not reference):
        try:
            appeal = _load_appeal(sample_name)
            reference = reference or appeal["seller_id"]
            if not supplied:
                extracted = extract_invoice_fields.invoke({"sample_name": sample_name})
                if "error" in extracted:
                    return extracted
                supplied = extracted["invoices"]
        except (FileNotFoundError, KeyError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}
    if not supplied:
        return {"error": "no invoices supplied", "tool_version": 1}

    sales = {r["sku"]: r for r in _read_rows("seller_sales.csv")
             if r["seller_id"] == reference}
    distributors = {r["supplier_name"].lower(): r
                    for r in _read_rows("authorized_distributors.csv")}

    findings, checks = [], []
    invoiced_by_sku: dict[str, int] = {}
    for invoice in supplied:
        if "error" in invoice:
            findings.append({"kind": "unreadable_document", "detail": invoice["error"],
                             "file": invoice.get("file", "")})
            continue

        supplier = invoice.get("supplier", "")
        entry = distributors.get(supplier.lower())
        status = entry["status"] if entry else "not_on_list"
        checks.append({"check": "supplier_on_authorized_list", "supplier": supplier,
                       "status": status})
        if status != "authorized":
            findings.append({
                "kind": "supplier_not_authorized", "supplier": supplier, "status": status,
                "detail": f"'{supplier}' is {status.replace('_', ' ')} on the sample "
                          f"distributor list.",
                "question": "Ask the seller for the authorisation letter from the brand."})

        arithmetic_total = 0.0
        for item in invoice.get("line_items", []):
            expected = round(item["quantity"] * item["unit_price"], 2)
            arithmetic_total += item["line_total"]
            checks.append({"check": "line_arithmetic", "sku": item["sku"],
                           "expected": expected, "stated": item["line_total"]})
            if abs(expected - item["line_total"]) > 0.01:
                findings.append({
                    "kind": "line_arithmetic_mismatch", "sku": item["sku"],
                    "expected": expected, "stated": item["line_total"],
                    "detail": f"{item['quantity']} x {item['unit_price']} is {expected}, "
                              f"but the line reads {item['line_total']}."})
            invoiced_by_sku[item["sku"]] = invoiced_by_sku.get(item["sku"], 0) + item["quantity"]

        stated = invoice.get("stated_total")
        if stated is not None and abs(arithmetic_total - stated) > 0.01:
            findings.append({
                "kind": "total_mismatch", "expected": round(arithmetic_total, 2),
                "stated": stated,
                "detail": f"Line items sum to {arithmetic_total:.2f}, the invoice states "
                          f"{stated:.2f}."})

        invoice_date = _date(invoice.get("invoice_date", ""))
        for sku in {i["sku"] for i in invoice.get("line_items", [])}:
            record = sales.get(sku)
            if not record:
                continue
            first_sale = _date(record["first_sale_date"])
            checks.append({"check": "invoice_precedes_first_sale", "sku": sku,
                           "invoice_date": invoice.get("invoice_date", ""),
                           "first_sale_date": record["first_sale_date"]})
            if invoice_date and first_sale and invoice_date > first_sale:
                findings.append({
                    "kind": "invoice_dated_after_first_sale", "sku": sku,
                    "invoice_date": invoice_date.isoformat(),
                    "first_sale_date": first_sale.isoformat(),
                    "detail": f"The invoice for {sku} is dated after the first recorded "
                              f"sale, so it cannot account for the earlier units.",
                    "question": "Ask for the earlier invoice covering the first sales."})

    for sku, record in sales.items():
        sold = int(record["units_sold"])
        invoiced = invoiced_by_sku.get(sku, 0)
        checks.append({"check": "units_invoiced_cover_units_sold", "sku": sku,
                       "units_invoiced": invoiced, "units_sold": sold})
        if invoiced < sold:
            findings.append({
                "kind": "units_sold_exceed_units_invoiced", "sku": sku,
                "units_invoiced": invoiced, "units_sold": sold,
                "shortfall": sold - invoiced,
                "detail": f"{sold} units of {sku} were sold but the documents account "
                          f"for {invoiced}.",
                "question": "Ask for the invoices covering the remaining units."})

    return {
        "seller_id": reference,
        "findings": findings, "finding_count": len(findings),
        "checks_run": checks, "check_count": len(checks),
        "units_invoiced_by_sku": invoiced_by_sku,
        "consistent": not findings,
        "note": ("Each finding is a discrepancy between documents, which is a question "
                 "to put to the seller rather than a conclusion about them."),
        "tool_version": 1,
    }


@tool(args_schema=AppealQualityArgs)
def appeal_quality(letter: str = "", sample_name: str = "") -> dict[str, Any]:
    """Check whether an appeal covers cause, correction, prevention and evidence.

    Structural checks on wording, which is crude: a section reported missing is a
    prompt to read the letter again, not a judgement of the seller's case.
    """
    body = letter
    if not body and sample_name:
        try:
            body = _load_appeal(sample_name).get("appeal_letter", "")
        except (FileNotFoundError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}
    if not body:
        return {"error": "no appeal letter supplied", "tool_version": 1}

    lowered = body.lower()
    sections = {}
    for name, markers in APPEAL_SECTIONS.items():
        hits = sorted({m for m in markers if m in lowered})
        sections[name] = {"present": bool(hits), "matched": hits}

    present = [name for name, entry in sections.items() if entry["present"]]
    missing = [name for name, entry in sections.items() if not entry["present"]]
    deflection = sorted({p for p in DEFLECTION_PHRASES if p in lowered})
    specifics = sorted(set(re.findall(r"\b[A-Z]{2,}-?[A-Z0-9]{2,}-?\d{2,}\b", body)))

    if not missing and not deflection:
        level = "APPEAL_WELL_SUPPORTED"
    elif len(present) <= 2 or (deflection and len(missing) >= 2):
        level = "APPEAL_WEAK"
    else:
        level = "NEEDS_MORE_EVIDENCE"

    return {
        "level": level,
        "sections": sections,
        "sections_present": present, "sections_missing": missing,
        "identifiers_cited": specifics,
        "deflection_phrases": deflection,
        "words": len(body.split()),
        "note": ("Structural checks on wording only. A missing section means the words "
                 "were not found, not that the seller has no case; read the letter."),
        "tool_version": 1,
    }

## 6. Prompt addenda and workflows

Sector guidance and the three workflow definitions.

In [ ]:
ADDENDA = {
    "coordinator": (
        "Risk signals are not proof. Describe evidence neutrally and give the specialist "
        "concrete verification steps they can carry out. Never accuse a customer or a seller, "
        "and never state as fact what the tools only observed. Every workflow here routes a "
        "case to a person; none of them decides it."
    ),
    "analyst": (
        "Report a score with the factors that produced it and the weight of each. Where a "
        "factor has an ordinary innocent explanation, say so in the same breath. A band or a "
        "score describes a case, never a person."
    ),
    "researcher": (
        "Policy statements must cite a knowledge source. Card network rules and deadlines vary "
        "by network and processor: attribute them, and say that the processor confirms them."
    ),
    "executor": (
        "Customer personal data: use only the fields the task needs, and never repeat full card "
        "numbers, email addresses, phone numbers or street addresses. The lookup tools return "
        "masked forms already; pass those through unchanged rather than reconstructing them."
    ),
}


class ReturnDisputeInputs(BaseModel):
    order_id: str = Field(default="", description="Order reference.")
    claim_type: str = Field(default="damaged",
                            description="damaged, not_as_described, missing_item, "
                                        "not_received or changed_mind.")
    claim_text: str = Field(default="", description="What the customer said happened.")
    chat_transcript: str = Field(default="", description="The conversation, if there is one.")
    sample_name: str = Field(default="return_high_value_reused_photo",
                             description="Bundled claim scenario.")


class ChargebackInputs(BaseModel):
    dispute_id: str = Field(default="")
    order_id: str = Field(default="")
    reason_category: str = Field(default="not_received",
                                 description="The dispute reason given by the issuer.")
    amount: float = Field(default=0.0)
    notice_date: str = Field(default="", description="ISO date the notice arrived.")
    response_window_days: int = Field(default=0, description="Zero means use the default.")
    sample_name: str = Field(default="dispute_not_received_strong")


class SellerAppealInputs(BaseModel):
    seller_id: str = Field(default="")
    violation_type: str = Field(default="counterfeit",
                                description="counterfeit, ip_complaint, performance or "
                                            "prohibited_item.")
    appeal_letter: str = Field(default="", description="The seller's appeal, as text.")
    sample_name: str = Field(default="appeal_invoice_mismatch")


RETURN_DISPUTE_PLAN = Plan(
    objective="Assess a return claim and route it to the right reviewer.",
    steps=[
        PlanStep(id="s1", agent="executor",
                 instruction="Look up the order and the customer's history, using masked "
                             "contact details only.",
                 tool_hints=["order_lookup", "customer_history"],
                 expected_output="order and account summary"),
        PlanStep(id="s2", agent="executor",
                 instruction="Check the claim against the return policy and name any "
                             "required evidence that is missing.",
                 tool_hints=["policy_check"], depends_on=["s1"],
                 expected_output="policy fit with the window used"),
        PlanStep(id="s3", agent="analyst",
                 instruction="Describe the supplied photos and the conversation, keeping "
                             "every observation neutral.",
                 tool_hints=["image_signals", "transcript_signals"], depends_on=["s1"],
                 expected_output="image and conversation signals"),
        PlanStep(id="s4", agent="analyst",
                 instruction="Score the claim from the documented weights and route it, "
                             "listing every factor that contributed.",
                 tool_hints=["risk_score", "route_claim"], depends_on=["s2", "s3"],
                 expected_output="a score with factors and a routing level"),
        PlanStep(id="s5", agent="researcher",
                 instruction="Find the policy clauses and past cases that bear on this claim.",
                 tool_hints=["search_knowledge"], expected_output="cited policy and cases"),
    ],
)

CHARGEBACK_PLAN = Plan(
    objective="Assemble a representment packet and say how complete it is.",
    steps=[
        PlanStep(id="s1", agent="executor",
                 instruction="Build the evidence checklist for this reason category and "
                             "work out the response deadline.",
                 tool_hints=["evidence_checklist", "response_deadline"],
                 expected_output="checklist and deadline"),
        PlanStep(id="s2", agent="executor",
                 instruction="Gather what the order record holds against that checklist.",
                 tool_hints=["gather_evidence"], depends_on=["s1"],
                 expected_output="gathered evidence"),
        PlanStep(id="s3", agent="analyst",
                 instruction="Score completeness, name what is missing, and give the "
                             "likelihood band with its reasons.",
                 tool_hints=["completeness_score", "win_likelihood_band"], depends_on=["s2"],
                 expected_output="completeness and a band"),
        PlanStep(id="s4", agent="researcher",
                 instruction="Draft the rebuttal letter citing the exhibits by number.",
                 tool_hints=["draft_rebuttal"], depends_on=["s2"],
                 expected_output="a draft letter"),
        PlanStep(id="s5", agent="researcher",
                 instruction="Find the dispute lifecycle guidance relevant to this reason.",
                 tool_hints=["search_knowledge"], expected_output="cited guidance"),
    ],
)

SELLER_APPEAL_PLAN = Plan(
    objective="Summarise a seller's appeal neutrally for an enforcement reviewer.",
    steps=[
        PlanStep(id="s1", agent="executor",
                 instruction="Summarise the seller's standing and the enforcement history.",
                 tool_hints=["seller_profile", "violation_timeline"],
                 expected_output="seller standing and timeline"),
        PlanStep(id="s2", agent="executor",
                 instruction="Extract the fields from the supplied invoices, naming any "
                             "field that could not be found.",
                 tool_hints=["extract_invoice_fields"], depends_on=["s1"],
                 expected_output="invoice fields"),
        PlanStep(id="s3", agent="analyst",
                 instruction="Compare the documents against the sales record and assess "
                             "whether the appeal covers cause, correction and prevention.",
                 tool_hints=["invoice_consistency", "appeal_quality"], depends_on=["s2"],
                 expected_output="discrepancies and appeal structure"),
        PlanStep(id="s4", agent="researcher",
                 instruction="Find the seller policy and comparable past appeal outcomes.",
                 tool_hints=["search_knowledge"], expected_output="cited policy and cases"),
    ],
)

# Demo-mode scripts. Arguments are fixed, which is why every tool accepts a sample
# name or falls back to what the previous step loaded.
RETURN_DISPUTE_SCRIPT = {
    "s1": [{"tool_calls": [
        {"name": "order_lookup", "args": {"sample_name": "return_high_value_reused_photo"}},
        {"name": "customer_history", "args": {"sample_name": "return_high_value_reused_photo"}}]},
        "looked up"],
    "s2": [{"tool_calls": [
        {"name": "policy_check", "args": {"sample_name": "return_high_value_reused_photo"}}]},
        "policy checked"],
    "s3": [{"tool_calls": [
        {"name": "image_signals", "args": {"sample_name": "return_high_value_reused_photo"}},
        {"name": "transcript_signals",
         "args": {"sample_name": "return_high_value_reused_photo"}}]}, "signals gathered"],
    "s4": [{"tool_calls": [
        {"name": "risk_score", "args": {"sample_name": "return_high_value_reused_photo"}},
        {"name": "route_claim", "args": {"sample_name": "return_high_value_reused_photo"}}]},
        "scored and routed"],
    "s5": [{"tool_calls": [{"name": "search_knowledge", "args": {
        "query": "return policy window evidence requirements claim review", "k": 4,
        "include_cases": True}}]}, "policy found"],
}

CHARGEBACK_SCRIPT = {
    "s1": [{"tool_calls": [
        {"name": "evidence_checklist", "args": {"sample_name": "dispute_not_received_strong"}},
        {"name": "response_deadline", "args": {"sample_name": "dispute_not_received_strong"}}]},
        "checklist built"],
    "s2": [{"tool_calls": [
        {"name": "gather_evidence", "args": {"sample_name": "dispute_not_received_strong"}}]},
        "evidence gathered"],
    "s3": [{"tool_calls": [
        {"name": "completeness_score", "args": {"sample_name": "dispute_not_received_strong"}},
        {"name": "win_likelihood_band",
         "args": {"sample_name": "dispute_not_received_strong"}}]}, "scored"],
    "s4": [{"tool_calls": [
        {"name": "draft_rebuttal", "args": {"sample_name": "dispute_not_received_strong"}}]},
        "drafted"],
    "s5": [{"tool_calls": [{"name": "search_knowledge", "args": {
        "query": "chargeback representment evidence delivery confirmation", "k": 4}}]},
        "guidance found"],
}

SELLER_APPEAL_SCRIPT = {
    "s1": [{"tool_calls": [
        {"name": "seller_profile", "args": {"sample_name": "appeal_invoice_mismatch"}},
        {"name": "violation_timeline", "args": {"sample_name": "appeal_invoice_mismatch"}}]},
        "profiled"],
    "s2": [{"tool_calls": [
        {"name": "extract_invoice_fields", "args": {"sample_name": "appeal_invoice_mismatch"}}]},
        "extracted"],
    "s3": [{"tool_calls": [
        {"name": "invoice_consistency", "args": {"sample_name": "appeal_invoice_mismatch"}},
        {"name": "appeal_quality", "args": {"sample_name": "appeal_invoice_mismatch"}}]},
        "compared"],
    "s4": [{"tool_calls": [{"name": "search_knowledge", "args": {
        "query": "seller policy counterfeit appeal invoice authorisation", "k": 4,
        "include_cases": True}}]}, "policy found"],
}

RETURN_DISPUTE_WORKFLOW = WorkflowSpec(
    id="ecommerce.return_dispute",
    name="Return and dispute assessment",
    description="Assess a return claim against policy and evidence, and route it for review.",
    input_schema=ReturnDisputeInputs,
    accepted_uploads=[".jpg", ".jpeg", ".png", ".txt"],
    step_template="1. order and history  2. policy fit  3. image and conversation signals  "
                  "4. score and route  5. gather policy",
    default_plan=RETURN_DISPUTE_PLAN,
    fake_script={"steps": RETURN_DISPUTE_SCRIPT},
    level_vocab=["FAST_TRACK_ELIGIBLE", "SPECIALIST_REVIEW", "HIGH_RISK_REVIEW"],
    forbidden_phrases=[r"\bcustomer is (?:lying|a fraudster|dishonest)\b",
                       r"\bconfirmed fraud\b", r"\bproves? (?:the )?fraud\b",
                       r"\bfraudulent claim\b"],
    sample_name="return_high_value_reused_photo",
    example_request="Assess this damage claim and tell me who should review it.",
)

CHARGEBACK_WORKFLOW = WorkflowSpec(
    id="ecommerce.chargeback_packet",
    name="Chargeback representment packet",
    description="Build a representment packet, score its completeness, and draft the rebuttal.",
    input_schema=ChargebackInputs,
    accepted_uploads=[".json", ".txt", ".pdf"],
    step_template="1. checklist and deadline  2. gather evidence  3. completeness and band  "
                  "4. draft rebuttal  5. gather guidance",
    default_plan=CHARGEBACK_PLAN,
    fake_script={"steps": CHARGEBACK_SCRIPT},
    level_vocab=["STRONG_PACKET", "WEAK_PACKET", "ACCEPT_LIABILITY_SUGGESTED"],
    forbidden_phrases=[r"\bsubmitted to (?:the )?(?:bank|issuer|processor)\b",
                       r"\bdispute (?:is )?won\b", r"\bwe will win\b",
                       r"\bchargeback (?:has been )?reversed\b"],
    sample_name="dispute_not_received_strong",
    example_request="Build the representment packet for this chargeback notice.",
)

SELLER_APPEAL_WORKFLOW = WorkflowSpec(
    id="ecommerce.seller_appeal",
    name="Seller appeal review",
    description="Summarise a seller's appeal and its supporting documents for a reviewer.",
    input_schema=SellerAppealInputs,
    accepted_uploads=[".txt", ".pdf", ".json"],
    step_template="1. seller standing and timeline  2. extract invoices  "
                  "3. consistency and appeal structure  4. gather policy",
    default_plan=SELLER_APPEAL_PLAN,
    fake_script={"steps": SELLER_APPEAL_SCRIPT},
    level_vocab=["APPEAL_WELL_SUPPORTED", "NEEDS_MORE_EVIDENCE", "APPEAL_WEAK"],
    forbidden_phrases=[r"\bseller is (?:guilty|innocent)\b",
                       r"\baccount (?:has been )?reinstated\b",
                       r"\bcounterfeit confirmed\b", r"\bwe have reinstated\b"],
    sample_name="appeal_invoice_mismatch",
    example_request="Review this seller's appeal and the invoices they supplied.",
)

## 7. Sector pack

Tool allowlists per role, and registration.

In [ ]:
SECTOR_PACK = SectorPack(
    **sector_identity(SECTOR_ID),
    tools=[
        ToolSpec(tool=order_lookup, roles=["executor"]),
        ToolSpec(tool=customer_history, roles=["executor"]),
        ToolSpec(tool=policy_check, roles=["executor"]),
        ToolSpec(tool=image_signals, roles=["analyst"]),
        ToolSpec(tool=transcript_signals, roles=["analyst"]),
        ToolSpec(tool=risk_score, roles=["analyst"]),
        ToolSpec(tool=route_claim, roles=["analyst"]),
        ToolSpec(tool=evidence_checklist, roles=["executor"]),
        ToolSpec(tool=gather_evidence, roles=["executor"]),
        ToolSpec(tool=response_deadline, roles=["executor"]),
        ToolSpec(tool=completeness_score, roles=["analyst"]),
        ToolSpec(tool=win_likelihood_band, roles=["analyst"]),
        ToolSpec(tool=draft_rebuttal, roles=["researcher"]),
        ToolSpec(tool=seller_profile, roles=["executor"]),
        ToolSpec(tool=violation_timeline, roles=["executor"]),
        ToolSpec(tool=extract_invoice_fields, roles=["executor"]),
        ToolSpec(tool=invoice_consistency, roles=["analyst"]),
        ToolSpec(tool=appeal_quality, roles=["analyst"]),
    ],
    workflows=[RETURN_DISPUTE_WORKFLOW, CHARGEBACK_WORKFLOW, SELLER_APPEAL_WORKFLOW],
    addenda=ADDENDA,
    ensure_samples=ensure_samples,
)

register_sector(SECTOR_PACK)

## Demo

Examples only; this cell is dropped from the built module.

In [ ]:
# Run every workflow offline and show what the reviewer would see.
import asyncio

ensure_samples()
for workflow in SECTOR_PACK.workflows:
    print(f"--- {workflow.id} ---")
    run_id = asyncio.run(start_run("ecommerce", workflow.id, workflow.example_request, {}))
    view = asyncio.run(get_run(run_id))
    print(view.status, view.brief["recommendation_level"] if view.brief else "no brief")

## Build check

Confirms the notebook reached the generated module.

In [ ]:
def hello() -> str:
    """Return this module's name, so the build pipeline can be checked end to end."""
    return "automatron_ecommerce"